# 02 · Preprocess by hand — filter, denoise (ICA), epoch

MOABB hides preprocessing. Here we do every step ourselves on raw PhysioNet EDF so the **order** and the **gotchas** stick:

montage → resample → band-pass → notch → **ICA** → re-reference → epoch → (autoreject).

Key rules: filter the *continuous* signal before epoching; fit ICA on a 1 Hz high-passed copy; anything that learns from data (ICA, autoreject) must later be fit inside CV folds (see notebook 06).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))
%load_ext autoreload
%autoreload 2
import numpy as np, matplotlib.pyplot as plt


In [ ]:
from eeglog.data import load_eegbci_raw
from eeglog import preprocess as pp
raw = load_eegbci_raw(subject=1, runs=[4, 8, 12])  # imagined L/R fist
raw

In [ ]:
# Before/after band-pass: PSD shows the band we keep (8–30 Hz).
raw_clean = pp.preprocess_raw(raw, l_freq=8., h_freq=30., do_ica=True)
fig = raw_clean.compute_psd(fmax=60).plot(show=False)

In [ ]:
# Inspect the ICA we fitted (stashed on the cleaned raw).
ica = raw_clean.info['temp']['ica']
print('excluded components:', ica.exclude)
fig = ica.plot_components(show=False) if ica.exclude else None

In [ ]:
# Epoch the cleaned continuous data, then get (X, y).
epochs = pp.make_epochs(raw_clean, tmin=0.5, tmax=3.5)
X, y = pp.epochs_to_xy(epochs)
print(X.shape, sorted(set(y)))
fig = epochs.average().plot(show=False)

**Checkpoint:** raw → cleaned → epoched, by hand. You saw the band we kept, the artifact components removed, and the evoked response.